In [1]:
from typing import Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

print("PyTorch version:", torch.__version__)

PyTorch version: 2.13.0+cpu


In [ ]:
"""
为什么随机丢弃激活可以缓解过拟合
    他们不一定让模型更加容易拟合训练集,而是希望模型学的更加稳健,能够迁移到未见的样本规律
"""

In [2]:
"""
Dropout 的数学形式
"""

x = torch.ones(10)
output = F.dropout(x, p=0.5)

print("Input:", x)
print("Output:", output)

Input: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
Output: tensor([2., 0., 0., 0., 2., 0., 0., 0., 0., 2.])


In [4]:
"""
为什么要除以保留概率?
"""

x = torch.tensor([1.0, 2.0, 3.0, 4.0])

outputs = torch.stack([F.dropout(x, p=0.5) for _ in range(1000)])
print("Input:", x)
print("Output:", outputs.mean(dim=0))

Input: tensor([1., 2., 3., 4.])
Output: tensor([1.0000, 2.0080, 3.1980, 4.1120])


In [7]:
"""
Dropout的PyTorch实现
    1. 推理模式下直接返回输入
    2. p=0时不丢弃任何元素
    3. 训练模式下采样mask,并处以保留率
"""


def dropout(x: Tensor, p: float = 0.5, training: bool = True) -> Tensor:
    if not 0.0 <= p <= 1.0:
        raise ValueError("p must be between 0.0 and 1.0")
    if not training or p == 0.0:
        return x

    if p == 1.0:
        return torch.zeros_like(x)

    keep = 1.0 - p
    mask = torch.rand_like(x) < keep
    return x * mask / keep


x = torch.arange(10, dtype=torch.float32)

actual = dropout(x, p=0.25)
excepted = F.dropout(x, p=0.25)

print('Custom dropout:', actual)
print('excepted dropout:', excepted)


class Dropout(nn.Module):
    def __init__(self, p: float = 0.5):
        super().__init__()
        if not 0.0 <= p < 1.0:
            raise ValueError("p must be between 0.0 and 1.0")
        self.p = p

    def forward(self, x: Tensor) -> Tensor:
        return dropout(x, p=self.p, training=self.training)

    def extra_repr(self) -> str:
        return f'p={self.p}'


Custom dropout: tensor([ 0.0000,  0.0000,  2.6667,  4.0000,  5.3333,  6.6667,  0.0000,  0.0000,
        10.6667, 12.0000])
excepted dropout: tensor([ 0.0000,  1.3333,  2.6667,  0.0000,  5.3333,  0.0000,  8.0000,  0.0000,
        10.6667, 12.0000])


In [10]:
"""
训练模式和推理模式
"""
dropout = nn.Dropout(p=0.5)
x = torch.ones(10)

dropout.train()
train_output1 = dropout(x)
train_output2 = dropout(x)

dropout.eval()
eval_output1 = dropout(x)
eval_output2 = dropout(x)

print("Training output1:", train_output1)
print("Training output2:", train_output2)
print("Evaluation output1:", eval_output1)
print("Evaluation output2:", eval_output2)

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(8, 2),
)

model.eval()

x = torch.randn(3, 4)

with torch.inference_mode():
    y = model(x)

print('y.shape', y.shape)

Training output1: tensor([2., 2., 2., 2., 2., 2., 0., 2., 2., 2.])
Training output2: tensor([2., 2., 2., 2., 0., 2., 0., 0., 2., 0.])
Evaluation output1: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
Evaluation output2: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
y.shape torch.Size([3, 2])


In [13]:
"""
Dropout 在反向传播中做了什么
"""

x = torch.ones(12, requires_grad=True)
y = F.dropout(x, p=0.5)
y.backward(gradient=torch.ones_like(y))

print("Dropout output:", y.data)
print("Input gradient:", x.grad)

Dropout output: tensor([2., 0., 2., 2., 2., 0., 2., 0., 2., 2., 2., 0.])
Input gradient: tensor([2., 0., 2., 2., 2., 0., 2., 0., 2., 2., 2., 0.])


In [14]:
"""
nn.Dropout 和F.Dropout
"""


class MLP1(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(256, output_dim),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)


# 使用函数形式
class MLP2(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, output_dim)

    def forward(self, x: Tensor) -> Tensor:
        x = self.fc1(x)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.fc2(x)
        return x


''

In [16]:
"""
Dropout1d,Dropout2d和Dropout3d
    随机丢弃的整个通道,而不是独立丢弃每个元素
"""

x = torch.ones(1, 4, 3, 3)
element_dropout = nn.Dropout(p=0.5)
channel_dropout = nn.Dropout2d(p=0.5)

element_output = element_dropout(x)
channel_output = channel_dropout(x)

print("Element dropout:", element_output)
print("Channel dropout:", channel_output)

Element dropout: tensor([[[[0., 0., 2.],
          [0., 0., 2.],
          [2., 2., 2.]],

         [[2., 0., 0.],
          [0., 0., 2.],
          [0., 0., 2.]],

         [[0., 2., 2.],
          [0., 0., 0.],
          [2., 0., 2.]],

         [[2., 0., 0.],
          [0., 2., 0.],
          [2., 0., 0.]]]])
Channel dropout: tensor([[[[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]],

         [[2., 2., 2.],
          [2., 2., 2.],
          [2., 2., 2.]],

         [[0., 0., 0.],
          [0., 0., 0.],
          [0., 0., 0.]]]])


In [ ]:
"""
Dropout应该放在哪里
    linear ->Activation ->Dropout->linear
    - p太小,正则化可能效果不明显
    - p 太大,信息丢失太多, 模型可能欠拟合
    - 网络太小,数据充足,通常不需要很强的dropout
    - 模型越容易过拟合.才更有理由增加dropout
    - 通常p在0.1-0.5之间,具体看数据表现
"""
black = nn.Sequential(
    nn.Linear(2, 3),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(2, 3),
)